# Phase 5: ML Model Exploration and Cross-Validation

This notebook loads persisted Phase 5 results from the executed runner. It does not fabricate outputs or evaluate final test runs.

## Objective and locked protocol

Models: Random Forest, Gradient Boosting, HistGradientBoosting, and XGBoost. Training runs are the ten locked training runs; validation is ID16 and ID22; final test runs ID18 and ID19 remain untouched. Model selection uses five-fold GroupKFold grouped by exp_id on training data only.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
results_dir = project_root / 'results' / 'freetwinev'
figures_dir = results_dir / 'figures'
summary = pd.read_csv(results_dir / 'phase5_results.csv')
per_run = pd.read_csv(results_dir / 'phase5_results_per_run.csv')
feature_importance = pd.read_csv(results_dir / 'phase5_feature_importance.csv')
predictions = pd.read_csv(results_dir / 'phase5_predictions_validation.csv')
config = json.loads((results_dir / 'phase5_config.json').read_text(encoding='utf-8'))
audit = json.loads((results_dir / 'phase5_audit.json').read_text(encoding='utf-8'))
print('Loaded persisted Phase 5 artifacts; audit:', audit['overall_status'])

## Feature definitions

Baseline has 9 physical telemetry features. Extended has 21 features, adding current, power, rolling-memory, thermal-gradient, and interaction terms. Targets are target_DeltaT_60s and target_DeltaT_300s. Absolute temperature is reconstructed as T_cell_max + predicted_DeltaT.

In [ ]:
for name, features in config['feature_sets'].items():
    print(name, len(features), features)
print(config['cv_type'], config['group_key'], config['cv_folds'])

## Hyperparameter search

The compact eight-configuration grids per model are persisted in phase5_config.json. MAE selects the refit configuration; RMSE and R2 are recorded independently across grouped folds.

In [ ]:
for model_name, grid in config['hyperparameter_grids'].items():
    print(model_name, grid)

## CV results

These are executed training-only grouped-fold results, not fabricated notebook outputs.

In [ ]:
summary[['Horizon', 'Model', 'Feature_Set', 'CV_MAE', 'CV_MAE_Std', 'CV_RMSE', 'CV_RMSE_Std', 'CV_R2', 'CV_R2_Std', 'Training_Time_s', 'Best_Params']]

## Final validation evaluation

Each configuration was selected from training-only CV and evaluated once on pooled ID16 and ID22. Absolute-temperature metrics are included in the persisted table.

In [ ]:
summary[['Horizon', 'Model', 'Feature_Set', 'Val_MAE', 'Val_RMSE', 'Val_R2', 'Val_Bias', 'Val_Max_AE', 'Val_Abs_MAE', 'Val_Abs_RMSE', 'Val_Abs_R2']]

## Per-run validation and feature importance

ID16 is the extreme-load regime and ID22 is the chilled/high-flow regime. Tree importance is diagnostic, not causal and not used for feature selection.

In [ ]:
per_run[['Horizon', 'Model', 'Feature_Set', 'Run', 'N', 'MAE', 'RMSE', 'R2', 'Bias', 'Max_AE', 'Abs_MAE', 'Abs_RMSE', 'Abs_R2']]

In [ ]:
feature_importance.sort_values(['Horizon', 'Feature_Set', 'Model', 'Importance_Impurity'], ascending=[True, True, True, False]).head(40)

## Diagnostic figures

The executed runner generated figures 21-30 under results/freetwinev/figures/. This cell displays representative persisted figures.

In [ ]:
paths = [figures_dir / n for n in ['21_actual_vs_predicted_60s.png', '23_residual_distributions.png', '26a_per_run_mae_60s.png', '27_model_comparison_horizons.png', '28a_feature_importance_60s.png', '29_cv_vs_val_mae.png']]
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for ax, path in zip(axes.flat, paths):
    ax.imshow(plt.imread(path)); ax.set_title(path.name); ax.axis('off')
plt.tight_layout()

## Audit and conclusion

The automated audit must be PASS, with ID18 and ID19 excluded. Phase 5 results do not show tree ensembles outperforming Phase 4 Extended Ridge: at 60s the best tree validation MAE is 0.2911 C versus Ridge 0.1598 C; at 300s the best tree validation RMSE is 6.3872 C versus Ridge 4.5423 C. ID16 drives the largest 300s generalization failures. Phase 5 ends here; final evaluation is reserved for the explicitly authorized next phase.

In [ ]:
assert audit['overall_status'] == 'PASS'
assert audit['test_runs_locked'] == ['ID18', 'ID19']
assert set(predictions['exp_id']).isdisjoint({'ID18', 'ID19'})
audit['overall_status']